# Using the functional API

In addition to the Object API (`Solver`), `qubo-solver` also exposes a functional API. It is slightly more verbose, but gives better insight and control: some things are only possible with the functional API.

## Classical approach

We'll start with the classical approach since the quantum one has some specificities

In [ ]:
from qubosolver import (
    Instance,
    solving,
    matrix,
    analysis,
    bitstrings,
    torch_rng,
)

Q = matrix.tensor([
    [-0.2, 0.0, 1.0],
    [ 0.0, 0.0, 1.5],
    [ 1.0, 1.5, 0.0],
])
instance = Instance(Q)

# Run tabu search in parallel from 5 starting bitstrings
start = bitstrings.rand(5, instance.size, rng=torch_rng(15))
solution = solving.tabu_search.solve(instance, start, time_limit=10.0)

print(analysis.to_dataframe([solution]))

## Quantum approach

In [ ]:
from qubosolver import (
    Instance,
    Solution,
    solving,
    embedding,
    drive_shaping,
    matrix,
    LocalEmulator,
    analysis,
)
import qoolqit

Q = matrix.tensor([
    [-0.2, 0.0, 1.0],
    [ 0.0, 0.0, 1.5],
    [ 1.0, 1.5, 0.0],
])
instance = Instance(Q)

device = qoolqit.AnalogDevice()
backend = LocalEmulator()

register = embedding.blade.embed(instance)
drive = drive_shaping.proportional_diagonal.build_drive(instance, register, device=device)
job = solving.analog_quantum_sampling.solve(register, drive, backend=backend, device=device)
solution = Solution.from_results(job.results(), instance)

print(analysis.to_dataframe([solution]))

In [ ]:
from qubosolver import (
    Instance,
    Solution,
    solving,
    embedding,
    drive_shaping,
    matrix,
    RemoteEmulator,
    analysis,
)
import qoolqit

import qoolqit
from qoolqit.execution import QPU
from pasqal_cloud import PasqalCloudConnection

# Replace with your username, project id and password on the Pasqal Cloud.
USERNAME="#TO_PROVIDE"
PROJECT_ID="#TO_PROVIDE"
PASSWORD=None

Q = matrix.tensor([
    [-0.2, 0.0, 1.0],
    [ 0.0, 0.0, 1.5],
    [ 1.0, 1.5, 0.0],
])
instance = Instance(Q)

if PASSWORD is not None:

    # Setup connection
    connection = PasqalCloudConnection(
        username=USERNAME,
        password=PASSWORD,
        project_id=PROJECT_ID,
    )

    emulate = True
    if emulate:
        device = qoolqit.AnalogDevice()
        backend = RemoteEmulator(connection=connection)
    else:
        print(f"Available devices: {connection.fetch_available_devices()}")
        device = qoolqit.Device.from_connection(connection, "FRESNEL_CAN1")
        backend = QPU(connection=connection, num_shots=1000)

    register = embedding.blade.embed(instance)
    drive = drive_shaping.proportional_diagonal.build_drive(instance, register, device=device)
    job = solving.analog_quantum_sampling.solve(register, drive, backend=backend, device=device)
    solution = Solution.from_results(job.results(), instance)

    print(analysis.to_dataframe([solution]))

Note about job --> async for remote, but also works in local. see qoolqit doc, and save/load tuto

In [ ]:
import pathlib
import json
from qoolqit.execution.job import get_batch_id, retrieve_remote_job, JobStatus

metadata = {
    "job_id": job.job_id(),
    "batch_id": get_batch_id(job),
}

output_directory = pathlib.Path.cwd() / "tmp" / "qubosolver-in-full" 
output_directory.mkdir(parents=True, exist_ok=True)
metadata_file = output_directory / "metadata.json"
data_file = output_directory / "data.bin"

with metadata_file.open("w") as f:
    json.dump(metadata, f)
with data_file.open("wb") as f:
    instance.save(f)

In [ ]:
with metadata_file.open("r") as f:
    metadata = json.load(f)
with data_file.open("rb") as f:
    reloaded_instance = Instance.load(f)

reloaded_job = retrieve_remote_job(connection, metadata["job_id"], batch_id=metadata["batch_id"])

status = reloaded_job.get_status()
print(f"Job status: {status}")

if status == JobStatus.DONE:
    solution = Solution.from_results(reloaded_job.results(), reloaded_instance)
    print(analysis.to_dataframe([solution]))